# Tetris RL — PPO 학습 (Colab)

`common.models.TetrisPolicyNet` 을 `common.env.TetrisPlacementEnv` 위에서 PPO로 학습합니다.
이 망으로 학습하면 결과 `.pt` 가 `netbot` 에 바로 로드되고 `netbot.export_onnx` 로
`model/bots/run.onnx`(C++ 인게임 봇용)로 변환됩니다 — 가중치 이식 불필요.

**런타임: GPU 권장** (런타임 → 런타임 유형 변경 → GPU). env 스텝은 CPU(SimGame)라
완전 GPU 바운드는 아니지만 망 업데이트가 빨라집니다.

## 1. 저장소 클론 + 빌드 의존성

In [ ]:
REPO_URL = 'https://github.com/<USER>/Tetris-Multiplayer-RL.git'   # ← 본인 fork 로 교체

import os
if not os.path.exists('Tetris-Multiplayer-RL'):
    !git clone $REPO_URL
%cd Tetris-Multiplayer-RL
!git pull

In [ ]:
!apt-get install -qq cmake g++ python3-dev > /dev/null
!pip install -q -r python/requirements-colab.txt
!rm -rf build
!rm -f python/sim/tetris_py*.so

## 2. pybind11 시뮬 모듈(`tetris_py`) 빌드 → `python/sim/` 배치

In [ ]:
import subprocess, shutil, glob

PYBIND11_DIR = subprocess.check_output(['python','-m','pybind11','--cmakedir']).decode().strip()
print('pybind11 cmake dir:', PYBIND11_DIR)

!mkdir -p build
!cd build && cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DTETRIS_BUILD_GAME=OFF \
    -DTETRIS_BUILD_PY=ON \
    -DTETRIS_BUILD_TEST=ON \
    -Dpybind11_DIR=$PYBIND11_DIR
!cmake --build build -j --target tetris_py sim_hash_dump

for so in glob.glob('build/tetris_py*.so'):
    print('copying', so, '-> python/sim/')
    shutil.copy(so, 'python/sim/')

## 3. 임포트 스모크 테스트 (sim + env)

In [ ]:
import sys
sys.path.insert(0, 'python')

from sim import SimGame
from common.env import TetrisPlacementEnv

env = TetrisPlacementEnv(seed=42)
obs, info = env.reset()
print('obs keys      :', {k: tuple(v.shape) for k, v in obs.items()})
print('legal actions :', int(info['legal_mask'].sum()), '/ 40')
obs, r, term, trunc, info = env.step(int(info['legal_mask'].argmax()))
print('step reward   :', r, ' game_over:', term)

## 4. PPO 학습

먼저 짧게(예: 20만 스텝) 돌려 학습이 도는지 확인 → 길게 늘리세요.
`--shaping-coef 0` 이면 순수 줄-삭제 보상(매우 희소), 기본 0.5 는 구멍/높이 패널티로
초반 학습을 돕습니다. 로그의 `lines/ep`(에피소드당 삭제 줄 수)가 오르면 학습 중.

In [ ]:
!cd python && python -m train.ppo_tetris \
    --steps 200000 \
    --rollout 2048 --epochs 4 --minibatch 256 \
    --lr 3e-4 --gamma 0.99 --lam 0.95 --clip 0.2 \
    --ent-coef 0.01 --shaping-coef 0.5 \
    --out checkpoints/run.pt --save-every 10

## 5. ONNX 변환 (C++ 인게임 봇 / netbot 용)

In [ ]:
!cd python && python -m netbot.export_onnx checkpoints/run.pt ../model/bots/run.onnx
!ls -la model/bots/run.onnx

## 6. 산출물 내려받기 / 사용

- `python/checkpoints/run.pt` — 학습된 체크포인트 (`netbot` 가 직접 로드)
- `model/bots/run.onnx` — C++ 인게임 봇(`Single vs Bot`) 로스터에 뜨는 파일

```python
from google.colab import files
files.download('python/checkpoints/run.pt')
files.download('model/bots/run.onnx')
```

**netbot 으로 접속 테스트** (로컬 PC에서):
```bash
cd python
python -m netbot.client --connect 127.0.0.1:7777 --policy checkpoints/run.pt
```

**인게임 봇**: `model/bots/*.onnx` 를 두고 `TETRIS_BUILD_BOT=ON` + ONNX Runtime 으로 빌드.

> 아키텍처(`TetrisPolicyNet`)를 바꾸면 `common/models.py` 의 `ARCH_VERSION` 을 꼭 올리세요 —
> 안 그러면 체크포인트 로드가 하드 에러로 거부됩니다(조용한 불일치 방지).